# ਪਾਠ 11 - ਏਜੰਟ-ਟੂ-ਏਜੰਟ (A2A) ਪ੍ਰੋਟੋਕੋਲ


## ਸੈਟਅਪ


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity

In [ ]:
import os
import asyncio
from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## A2A ਪ੍ਰੋਟੋਕੋਲ ਕੀ ਹੈ?

ਇਹ **ਏਜੰਟ-ਤੋਂ-ਏਜੰਟ (A2A) ਪ੍ਰੋਟੋਕੋਲ** ਇੱਕ ਖੁੱਲ੍ਹਾ ਮਿਆਰ ਹੈ ਜੋ AI ਏਜੰਟਾਂ ਨੂੰ ਸੰਚਾਰ ਕਰਨ,
ਇਕ ਦੂਜੇ ਨੂੰ ਲੱਭਣ, ਅਤੇ ਸਹਿਯੋਗ ਕਰਨ — ਭਾਵੇਂ ਉਹ ਵੱਖ-ਵੱਖ ਫਰੇਮਵਰਕਾਂ 'ਤੇ ਬਣੇ ਹੋਣ ਜਾਂ ਹੋਸਟ
ਵੱਖ-ਵੱਖ ਸੇਵਾਵਾਂ ਦੁਆਰਾ।

Key concepts:

- **ਖੋਜ** – ਏਜੰਟ ਆਪਣੀਆਂ ਯੋਗਤਾਵਾਂ ਦਾ ਵਰਣਨ ਕਰਨ ਵਾਲਾ *ਏਜੰਟ ਕਾਰਡ* ਪ੍ਰਕਾਸ਼ਿਤ ਕਰਦੇ ਹਨ, ਇਸ ਨਾਲ
  ਹੋਰ ਏਜੰਟਾਂ (ਜਾਂ ਸੰਚਾਲਕਾਂ) ਲਈ ਕਿਸੇ ਕੰਮ ਲਈ ਸਹੀ ਮਾਹਿਰ ਨੂੰ ਲੱਭਣਾ ਆਸਾਨ ਹੋ ਜਾਂਦਾ ਹੈ।
- **ਸੁਨੇਹਾ-ਪ੍ਰਸਾਰ** – ਏਜੰਟ ਇੱਕ ਸਾਂਝੇ ਪ੍ਰੋਟੋਕੋਲ ਰਾਹੀਂ ਬਣਤਰਬੱਧ ਸੁਨੇਹਿਆਂ ਦਾ ਬਦਲਾਵ ਕਰਦੇ ਹਨ, ਤਾਂ ਜੋ ਇੱਕ
  ਬੇਨਤੀ ਨੂੰ ਦੂਜੇ ਦੁਆਰਾ ਸਮਝਿਆ ਅਤੇ ਪੂਰਾ ਕੀਤਾ ਜਾ ਸਕੇ, ਭਾਵੇਂ ਇਸਦੀ ਅੰਦਰੂਨੀ
  ਲਾਗੂ ਕਰਨ ਦੀ ਵਿਧੀ ਤੋਂ ਬਿਨਾਂ।
- **ਟਾਸਕ ਜੀਵਨਚੱਕਰ** – A2A ਅਜੇਹੀਆਂ ਹਾਲਤਾਂ ਨੂੰ ਪਰਿਭਾਸ਼ਿਤ ਕਰਦਾ ਹੈ ਜਿਵੇਂ ਕਿ *submitted*, *working*, *completed*, ਅਤੇ
  *failed*, ਸੰਚਾਲਕ ਨੂੰ ਇਹ ਪੂਰੀ ਨਜ਼ਰ ਦਿੰਦਾ ਹੈ ਕਿ ਸौंਪੀ ਗਈ ਟਾਸਕ ਕਿਵੇਂ ਤਰੱਕੀ ਕਰ ਰਹੀ ਹੈ।

ਇਸ ਪਾਠ ਵਿੱਚ ਅਸੀਂ A2A-ਸਟਾਈਲ ਸਹਿਯੋਗ ਦੀ ਨਕਲ ਕਰਦੇ ਹਾਂ ਤਿੰਨ ਵਿਸ਼ੇਸ਼ਤ ਯਾਤਰਾ ਏਜੰਟਾਂ ਨੂੰ
ਇੱਕ ਵਰਕਫਲੋ ਵਿੱਚ ਜੋੜ ਕੇ ਜਿੱਥੇ ਹਰ ਏਜੰਟ ਆਪਣਾ ਤਜਰਬਾ ਪੇਸ਼ ਕਰਦਾ ਅਤੇ ਨਤੀਜੇ ਅਗਲੇ ਨੂੰ ਭੇਜਦਾ ਹੈ।


## ਖਾਸ ਤਰ੍ਹਾਂ ਦੇ ਯਾਤਰਾ ਏਜੰਟ ਤਿਆਰ ਕਰਨਾ


In [ ]:
currency_agent = await provider.create_agent(
    name="CurrencyExchangeAgent",
    instructions="""You are a currency exchange specialist. You help travelers understand:
- Current exchange rates between currencies
- Best times to exchange money
- Tips for getting the best rates
When asked about a destination, provide relevant currency information.""",
)

activity_agent = await provider.create_agent(
    name="ActivityPlannerAgent",
    instructions="""You are a local activities specialist. You recommend:
- Must-see attractions and hidden gems
- Local experiences and cultural activities
- Restaurant and dining recommendations
Tailor suggestions to the traveler's interests.""",
)

travel_manager = await provider.create_agent(
    name="TravelManagerAgent",
    instructions="""You are a travel manager who coordinates between specialist agents.
When planning a trip:
1. Gather currency information from the currency specialist
2. Get activity recommendations from the activity planner
3. Synthesize everything into a cohesive travel brief
Present the final plan in an organized, easy-to-read format.""",
)

## ਵਰਕਫਲੋ ਰਾਹੀਂ ਬਹੁ-ਏਜੰਟ ਸਹਿਯੋਗ

ਅਸੀਂ ਤਿੰਨ ਏਜੰਟਾਂ ਨੂੰ ਇੱਕ ਲੜੀਵਾਰ ਵਰਕਫਲੋ ਵਿੱਚ ਜੋੜਦੇ ਹਾਂ ਜੋ A2A ਸੁਨੇਹਾ-ਪ੍ਰਸਾਰਣ ਦੀ ਨਕਲ ਕਰਦਾ ਹੈ:

1. **CurrencyExchangeAgent** ਉਪਭੋਗਤਾ ਦੀ ਬੇਨਤੀ ਪ੍ਰਾਪਤ ਕਰਦਾ ਹੈ ਅਤੇ ਮੁਦਰਾ ਸਲਾਹ ਦਿੰਦਾ ਹੈ.
2. **ActivityPlannerAgent** ਸੰਵਰਧਿਤ ਸੰਦਰਭ ਪ੍ਰਾਪਤ ਕਰਦਾ ਹੈ ਅਤੇ ਸਰਗਰਮੀਆਂ ਲਈ ਸਿਫਾਰਿਸ਼ਾਂ ਜੋੜਦਾ ਹੈ.
3. **TravelManagerAgent** ਦੋਹਾਂ ਇਨਪੁੱਟਾਂ ਨੂੰ ਮਿਲਾ ਕੇ ਆਖ਼ਰੀ ਯਾਤਰਾ ਸੰਖੇਪ ਤਿਆਰ ਕਰਦਾ ਹੈ.


In [ ]:
workflow = WorkflowBuilder(start_executor=currency_agent) \
    .add_edge(currency_agent, activity_agent) \
    .add_edge(activity_agent, travel_manager) \
    .build()

last_author = None
events = workflow.run(
    "Plan a week-long trip to Tokyo. I love food, temples, and technology.",
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## ਪ੍ਰੋਡਕਸ਼ਨ ਵਿੱਚ A2A ਦੀ ਸਮਝ

ਇੱਕ ਪ੍ਰੋਡਕਸ਼ਨ ਵਾਤਾਵਰਣ ਵਿੱਚ A2A ਪ੍ਰੋਟੋਕੋਲ ਸ਼ਕਤੀਸ਼ਾਲੀ ਕ੍ਰਾਸ-ਸਰਵਿਸ ਸਿਨਾਰੀਓਜ਼ ਨੂੰ ਸਮਭਵ ਬਣਾਉਂਦਾ ਹੈ:

| ਸਮਰੱਥਾ | ਵੇਰਵਾ |
|---|---|
| **ਕ੍ਰਾਸ-ਫ੍ਰੇਮਵਰਕ ਇੰਟਰਓਪਰੇਬਿਲਿਟੀ** | ਇੱਕ ਫ੍ਰੇਮਵਰਕ ਨਾਲ ਬਣਿਆ ਏਜੰਟ ਕਿਸੇ ਵੀ ਹੋਰ A2A-ਅਨੁਕੂਲ ਫ੍ਰੇਮਵਰਕ ਨਾਲ ਬਣੇ ਏਜੰਟ ਨੂੰ ਟਾਸਕ ਸੌਂਪ ਸਕਦਾ ਹੈ, ਜੋ ਸੱਚੀ ਅੰਤਰ-ਸੰਗਠਨ ਪਰਸਪਰਕਰਨ ਯੋਗਤਾ ਨੂੰ ਯਕੀਨੀ ਬਣਾਉਂਦਾ ਹੈ। |
| **ਸੇਵਾ-ਸੀਮਾਵਾਂ** | ਏਜੰਟ ਵੱਖ-ਵੱਖ ਮਾਈਕਰੋਸਰਵਿਸਜ਼, ਕਲਾਉਡ ਖੇਤਰਾਂ ਜਾਂ ਇੱਥੋਂ ਤੱਕ ਕਿ ਵੱਖ-ਵੱਖ ਸੰਸਥਾਵਾਂ ਵਿੱਚ ਰਹਿ ਸਕਦੇ ਹਨ, ਅਤੇ ਫਿਰ ਵੀ ਬੇਹੱਦ ਸੁਚਾਰੂ ਤਰੀਕੇ ਨਾਲ ਸਹਿਯੋਗ ਕਰ ਸਕਦੇ ਹਨ। |
| **ਡਾਇਨਾਮਿਕ ਖੋਜ** | ਇੱਕ ਓਰਕੈਸਟ੍ਰੇਟਰ ਰਨਟਾਈਮ 'ਤੇ Agent Card ਰਜਿਸਟਰੀ ਨੂੰ ਪੁੱਛ-ਤਾਛ ਕਰ ਸਕਦਾ ਹੈ ਤਾਂ ਕਿ ਦਿੱਤੇ ਗਏ ਉਪ-ਟਾਸਕ ਲਈ ਸਭ ਤੋਂ ਉਚਿਤ ਮਾਹਿਰ ਨੂੰ ਲੱਭ ਸਕੇ। |
| **ਸਟ੍ਰੀਮਿੰਗ & ਪੁਸ਼ ਨੋਟੀਫਿਕੇਸ਼ਨ** | A2A ਰੀਅਲ-ਟਾਈਮ ਪ੍ਰਗਤੀ ਅੱਪਡੇਟਾਂ ਲਈ Server-Sent Events (SSE) ਅਤੇ ਲੰਬੇ ਸਮੇਂ ਚੱਲਣ ਵਾਲੇ ਟਾਸਕਾਂ ਲਈ ਪੁਸ਼ ਨੋਟੀਫਿਕੇਸ਼ਨਾਂ ਨੂੰ ਸਮਰਥਨ ਕਰਦਾ ਹੈ। |

The workflow we built above is a simplified, in-process version of this pattern. In a real
deployment each agent would expose an HTTP endpoint, publish an Agent Card, and communicate
via the A2A JSON-RPC protocol.


## ਸਾਰ

ਇਸ ਪਾਠ ਵਿੱਚ ਤੁਸੀਂ ਸਿੱਖਿਆ:

1. **A2A ਪ੍ਰੋਟੋਕੋਲ ਕੀ ਹੈ** — ਏਜੰਟ-ਟੂ-ਏਜੰਟ ਖੋਜ, ਸੁਨੇਹਾ ਭੇਜਣਾ,
   ਅਤੇ ਕਾਰਜ ਪ੍ਰਬੰਧਨ।
2. **ਕਿਵੇਂ ਵਿਸ਼ੇਸ਼ ਏਜੰਟ ਬਣਾਉਣੇ ਹਨ** — ਇੱਕ ਕਰੰਸੀ ਐਕਸਚੇਂਜ ਏਜੰਟ, ਇੱਕ ਐਕਟਿਵਿਟੀ ਪਲੈਨਰ ਏਜੰਟ,
   ਅਤੇ ਇੱਕ ਟ੍ਰੈਵਲ ਮੈਨੇਜਰ ਔਰਕੇਸਟਰ।
3. **ਏਜੰਟਾਂ ਨੂੰ ਵਰਕਫਲੋ ਵਿੱਚ ਜੋੜਨ ਦਾ ਤਰੀਕਾ** — `WorkflowBuilder` ਦੀ ਵਰਤੋਂ ਕਰਕੇ ਲੜੀਵਾਰ
   ਏਜੰਟਾਂ ਦਰਮਿਆਨ ਸੁਨੇਹਾ-ਪਾਸਿੰਗ ਦਾ ਮਾਡਲ ਬਣਾਉਣ ਲਈ।
4. **ਉਤਪਾਦਨ ਵਿੱਚ A2A ਕਿਵੇਂ ਕੰਮ ਕਰਦਾ ਹੈ** — ਕ੍ਰਾਸ-ਫਰੇਮਵਰਕ, ਕ੍ਰਾਸ-ਸੇਵਾ ਸਹਿਯੋਗ ਨੂੰ ਯੋਗ ਬਣਾਉਂਦਾ ਹੈ
   ਡਾਇਨਾਮਿਕ ਖੋਜ ਅਤੇ ਸਟਰੀਮਿੰਗ ਅੱਪਡੇਟਸ ਨਾਲ।


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
ਦਿਸਕਲੇਮਰ:
ਇਸ ਦਸਤਾਵੇਜ਼ ਦਾ ਅਨੁਵਾਦ AI ਅਨੁਵਾਦ ਸੇਵਾ [Co-op Translator](https://github.com/Azure/co-op-translator) ਦੀ ਵਰਤੋਂ ਕਰਕੇ ਕੀਤਾ ਗਿਆ ਹੈ। ਜਿੰਨਾ ਸਮਭਵ ਹੈ ਅਸੀਂ ਸਹੀਅਤ ਲਈ ਕੋਸ਼ਿਸ਼ ਕਰਦੇ ਹਾਂ, ਪਰ ਕਿਰਪਾ ਕਰਕੇ ਧਿਆਨ ਰੱਖੋ ਕਿ ਸਵੈਚਾਲਿਤ ਅਨੁਵਾਦਾਂ ਵਿੱਚ ਤ੍ਰੁਟੀਆਂ ਜਾਂ ਅਣਸੁੱਧਤੀਆਂ ਹੋ ਸਕਦੀਆਂ ਹਨ। ਮੂਲ ਭਾਸ਼ਾ ਵਿੱਚ ਮੌਜੂਦ ਦਸਤਾਵੇਜ਼ ਨੂੰ ਹੀ ਪ੍ਰਮਾਣਿਕ ਸਰੋਤ ਮੰਨਿਆ ਜਾਣਾ ਚਾਹੀਦਾ ਹੈ। ਮਹੱਤਵਪੂਰਨ ਜਾਣਕਾਰੀ ਲਈ ਪੇਸ਼ੇਵਰ ਮਨੁੱਖੀ ਅਨੁਵਾਦ ਦੀ ਸਿਫਾਰਸ਼ ਕੀਤੀ ਜਾਂਦੀ ਹੈ। ਇਸ ਅਨੁਵਾਦ ਦੀ ਵਰਤੋਂ ਨਾਲ ਹੋਣ ਵਾਲੀਆਂ ਕਿਸੇ ਵੀ ਗਲਤਫਹਮੀਆਂ ਜਾਂ ਗਲਤ ਵਿਆਖਿਆਵਾਂ ਲਈ ਅਸੀਂ ਜ਼ਿੰਮੇਵਾਰ ਨਹੀਂ ਹਾਂ।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
